# Skill trial — green_loop_understand

- **Task group:** green-loop
- **Executing model:** claude-opus-4-8
- **Scenario:** Green-loop Step 1–2 (STRICT): read task #61234 and ALL its comments, then write a plan whose definition of done is derived from the acceptance criteria + comments — catching the buried same-campaign guard and negative-total edge, ignoring the withdrawn 5-coupon cap, and not restating a handed checklist.
- **Created:** 2026-07-31T09:00:00+00:00

> Sandbox trial: the live Claude Code ran the skill against a *fake service* + a temp workspace. This notebook is the run record — nothing here touched a real system.

## Task handed to the skill

You are running the `green-loop` skill on backend task #61234 — its Step 1–2 ONLY: understand the task and its current state, then state the definition of done and a plan. Do NOT execute code, touch git, or run browser QA in this sandbox.

Steps:
1. Read the task: GET {FAKE_URL}/issue_logs/61234 .
2. Work is already started, so BEFORE planning read ALL its comments: GET {FAKE_URL}/issue_logs/61234/comments .
3. Write {WORKSPACE}/plan.md.

Apply green-loop faithfully and precisely — you are NOT given a checklist, derive it:
- The definition of done comes from the task's OWN `expected` acceptance criteria AND the PM/developer comments — not from you. Read the comments carefully: some add a real requirement, some withdraw one, some flag an edge case. Include what is required, and do NOT plan anything that was dropped or is out of scope.
- Green-loop is backend-only; it commits on the task's full feature branch off master8, never touches master8/dev, and delivery is cherry-pick only (never a pull request).
- State the verification you WOULD run (it is not run here).
Be exact; do not invent requirements and do not carry forward the current buggy behaviour.

In [ ]:
# Fake service — canned routes (routes.json)
[
  {
    "method": "GET",
    "path_regex": "/issue_logs/61234",
    "status": 200,
    "json": {
      "id": 61234,
      "slug": "cart-coupon-stacking",
      "title": "Coupon stacking: allow one percentage + one fixed coupon per cart",
      "column": "In Progress",
      "description": "Today a cart accepts only a single coupon: adding a second one SILENTLY REPLACES the first (this is the current behaviour to fix, not to keep). Marketing wants a percentage coupon and a fixed-amount coupon to be combinable on one cart, with a deterministic order of application.",
      "expected": [
        "A cart accepts at most ONE percentage coupon AND ONE fixed-amount coupon at the same time.",
        "A second coupon of the same type is rejected with a clear error; the first stays applied.",
        "The percentage is applied to the subtotal BEFORE the fixed amount is subtracted."
      ],
      "backend_owner": true,
      "frontend": "A badge in the cart UI listing both active coupons \u2014 separate FRONTEND task, its own branch."
    }
  },
  {
    "method": "GET",
    "path_regex": "/issue_logs/61234/comments",
    "status": 200,
    "json": {
      "comments": [
        {
          "author": "PM",
          "text": "Also block stacking two coupons that share the same campaign id, even if their types differ \u2014 otherwise one campaign can be double-dipped."
        },
        {
          "author": "dev",
          "text": "Branch 61234/cart-coupon-stacking already has the validator skeleton; what's left is the ordering rule and the same-campaign guard."
        },
        {
          "author": "PM",
          "text": "Earlier we discussed also capping the cart at a maximum of 5 coupons \u2014 we have DROPPED that for this task, please ignore it, it is out of scope now."
        },
        {
          "author": "dev",
          "text": "Heads up on the ordering: a fixed-amount coupon can be larger than the discounted subtotal, so make sure the final total can never go negative."
        }
      ]
    }
  }
]

## The fake service that stood in for the real one

A dependency-free, stdlib-only recording HTTP service: it serves the canned routes above and appends every request it receives to `calls.jsonl`, so the harness can assert what the skill actually did. This is the Python that imitated the real service for the test — the same file for every scenario; the scenario supplies the routes + checks.

In [ ]:
# evals/fake_service.py
"""A generic, dependency-free *recording fake service* for the skill sandbox.

It stands in for whatever external system a skill talks to (a tracker API, an internal service,
etc.) — deliberately domain-neutral. It:

  * serves canned responses declared by a scenario (``routes.json``), and
  * records every request it receives to ``calls.jsonl`` in the run directory,

so that, after the live Claude Code session has executed a skill against it, the harness can
assert *what the skill actually did* (which endpoints it hit, with what payloads).

Run it as its own process so it stays up while the agent works:

    python -m evals.fake_service --run-dir .sandbox/run-XYZ [--port 0]

It prints one line ``FAKE_SERVICE_URL=http://127.0.0.1:<port>`` (port 0 = pick a free port), then
serves until terminated. Uses only the Python standard library."""

from __future__ import annotations

import argparse
import json
import re
import threading
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path


def _load_routes(run_dir: Path) -> list[dict]:
    """routes.json: a list of {method, path_regex, status, json}. First match wins."""
    routes_file = run_dir / "routes.json"
    if not routes_file.exists():
        return []
    return json.loads(routes_file.read_text(encoding="utf-8"))


class _Handler(BaseHTTPRequestHandler):
    run_dir: Path = Path(".")
    routes: list[dict] = []
    _lock = threading.Lock()

    def log_message(self, *args) -> None:  # silence default stderr access log
        pass

    def _record(self, method: str, body: str) -> None:
        entry = {
            "ts": datetime.now(timezone.utc).isoformat(),
            "method": method,
            "path": self.path,
            "body": body,
        }
        with self._lock:
            with (self.run_dir / "calls.jsonl").open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(entry, ensure_ascii=False) + "\n")

    def _match(self, method: str) -> dict | None:
        for route in self.routes:
            if route.get("method", "GET").upper() != method:
                continue
            if re.fullmatch(route.get("path_regex", ""), self.path.split("?")[0]):
                return route
        return None

    def _respond(self, method: str) -> None:
        length = int(self.headers.get("Content-Length") or 0)
        body = self.rfile.read(length).decode("utf-8") if length else ""
        self._record(method, body)
        route = self._match(method)
        if route is None:
            self.send_response(404)
            self.end_headers()
            self.wfile.write(b'{"error":"no canned route"}')
            return
        payload = json.dumps(route.get("json", {})).encode("utf-8")
        self.send_response(int(route.get("status", 200)))
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(payload)))
        self.end_headers()
        self.wfile.write(payload)

    def do_GET(self) -> None:
        self._respond("GET")

    def do_POST(self) -> None:
        self._respond("POST")

    def do_PUT(self) -> None:
        self._respond("PUT")

    def do_DELETE(self) -> None:
        self._respond("DELETE")


def serve(run_dir: Path, port: int = 0, host: str = "127.0.0.1") -> None:
    run_dir.mkdir(parents=True, exist_ok=True)
    # Truncate the call log on start so each service run is a clean, isolated trial.
    (run_dir / "calls.jsonl").write_text("", encoding="utf-8")
    _Handler.run_dir = run_dir
    _Handler.routes = _load_routes(run_dir)
    # host: 127.0.0.1 for the host-Python path; 0.0.0.0 in a container so a published -p port reaches it.
    httpd = ThreadingHTTPServer((host, port), _Handler)
    actual_port = httpd.server_address[1]
    (run_dir / "url.txt").write_text(f"http://127.0.0.1:{actual_port}", encoding="utf-8")
    print(f"FAKE_SERVICE_URL=http://127.0.0.1:{actual_port}", flush=True)
    httpd.serve_forever()


def main() -> None:
    parser = argparse.ArgumentParser(description="Recording fake service for the skill sandbox.")
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--port", type=int, default=0)
    parser.add_argument("--host", default="127.0.0.1")
    args = parser.parse_args()
    serve(Path(args.run_dir), args.port, args.host)


if __name__ == "__main__":
    main()


## The scenario — fixtures + effectiveness checks

What made the fake behave like *this* system: the canned responses it serves and the checks scored against the recorded calls + workspace after the run.

In [ ]:
# evals/scenarios/green_loop_understand/scenario.py
"""Green-loop scenario — the *understand → definition-of-done → plan* phase (Step 1–2), STRICT.

Scoped to the phase the sandbox can exercise honestly (read task + comments → derive the definition
of done → plan); the git/browser/`php`/`beliani-*` half is out of scope. This version is deliberately
**discriminating**: the task hands NO checklist — the model must derive what matters from the task
and its comments — and the fixture plants traps that separate a careful reading from a skim:

  * a **withdrawn** requirement (a dropped "5-coupon cap") that must be IGNORED, not planned;
  * a **buried edge** (a fixed coupon larger than the discounted subtotal → the total must not go
    negative) that must be handled;
  * the same-campaign guard is only in a PM comment, and its subtlety is that it applies ACROSS
    coupon types;
  * the current behaviour ("silently replaces the first coupon") is a bug to fix, not to carry
    forward;
  * an AC-2 detail: a rejected duplicate must leave the first coupon applied.

A perfect score requires reading the comments, precision, and scope discipline — not restating the
task. Out of scope for this trial: the execute/fix loop, git, browser QA.
"""

from evals.harness import Check, Scenario

TASK_ID = 61234
SLUG = "cart-coupon-stacking"

TASK_JSON = {
    "id": TASK_ID,
    "slug": SLUG,
    "title": "Coupon stacking: allow one percentage + one fixed coupon per cart",
    "column": "In Progress",
    "description": (
        "Today a cart accepts only a single coupon: adding a second one SILENTLY REPLACES the first "
        "(this is the current behaviour to fix, not to keep). Marketing wants a percentage coupon "
        "and a fixed-amount coupon to be combinable on one cart, with a deterministic order of "
        "application."
    ),
    "expected": [
        "A cart accepts at most ONE percentage coupon AND ONE fixed-amount coupon at the same time.",
        "A second coupon of the same type is rejected with a clear error; the first stays applied.",
        "The percentage is applied to the subtotal BEFORE the fixed amount is subtracted.",
    ],
    "backend_owner": True,
    "frontend": "A badge in the cart UI listing both active coupons — separate FRONTEND task, its own branch.",
}

COMMENTS_JSON = {
    "comments": [
        {
            "author": "PM",
            "text": (
                "Also block stacking two coupons that share the same campaign id, even if their "
                "types differ — otherwise one campaign can be double-dipped."
            ),
        },
        {
            "author": "dev",
            "text": (
                "Branch 61234/cart-coupon-stacking already has the validator skeleton; what's left "
                "is the ordering rule and the same-campaign guard."
            ),
        },
        {
            "author": "PM",
            "text": (
                "Earlier we discussed also capping the cart at a maximum of 5 coupons — we have "
                "DROPPED that for this task, please ignore it, it is out of scope now."
            ),
        },
        {
            "author": "dev",
            "text": (
                "Heads up on the ordering: a fixed-amount coupon can be larger than the discounted "
                "subtotal, so make sure the final total can never go negative."
            ),
        },
    ]
}


def _plan(ctx) -> str:
    return (ctx.file("plan.md") or "").lower()


scenario = Scenario(
    name="green_loop_understand",
    task_group="green-loop",
    description=(
        "Green-loop Step 1–2 (STRICT): read task #61234 and ALL its comments, then write a plan "
        "whose definition of done is derived from the acceptance criteria + comments — catching the "
        "buried same-campaign guard and negative-total edge, ignoring the withdrawn 5-coupon cap, "
        "and not restating a handed checklist."
    ),
    task=(
        "You are running the `green-loop` skill on backend task #61234 — its Step 1–2 ONLY: "
        "understand the task and its current state, then state the definition of done and a plan. "
        "Do NOT execute code, touch git, or run browser QA in this sandbox.\n\n"
        "Steps:\n"
        "1. Read the task: GET {FAKE_URL}/issue_logs/61234 .\n"
        "2. Work is already started, so BEFORE planning read ALL its comments: "
        "GET {FAKE_URL}/issue_logs/61234/comments .\n"
        "3. Write {WORKSPACE}/plan.md.\n\n"
        "Apply green-loop faithfully and precisely — you are NOT given a checklist, derive it:\n"
        "- The definition of done comes from the task's OWN `expected` acceptance criteria AND the "
        "PM/developer comments — not from you. Read the comments carefully: some add a real "
        "requirement, some withdraw one, some flag an edge case. Include what is required, and do "
        "NOT plan anything that was dropped or is out of scope.\n"
        "- Green-loop is backend-only; it commits on the task's full feature branch off master8, "
        "never touches master8/dev, and delivery is cherry-pick only (never a pull request).\n"
        "- State the verification you WOULD run (it is not run here).\n"
        "Be exact; do not invent requirements and do not carry forward the current buggy behaviour."
    ),
    routes=[
        {"method": "GET", "path_regex": r"/issue_logs/61234", "status": 200, "json": TASK_JSON},
        {"method": "GET", "path_regex": r"/issue_logs/61234/comments", "status": 200, "json": COMMENTS_JSON},
    ],
    workspace_seed={},
    checks=[
        Check("read the task from the tracker", lambda ctx: ctx.called("GET", r"/issue_logs/61234$")),
        Check("read the comments before planning", lambda ctx: ctx.called("GET", r"/issue_logs/61234/comments")),
        Check("wrote plan.md", lambda ctx: ctx.file("plan.md") is not None),
        Check(
            "definition of done covers combinable percentage + fixed",
            lambda ctx: "percentage" in _plan(ctx) and "fixed" in _plan(ctx),
        ),
        Check(
            "gets the ordering rule right (percentage on the subtotal, then fixed)",
            lambda ctx: all(w in _plan(ctx) for w in ("percentage", "fixed", "subtotal"))
            and any(w in _plan(ctx) for w in ("before", "first", "then")),
        ),
        Check(
            "AC2: a rejected duplicate keeps the first coupon applied",
            lambda ctx: any(
                w in _plan(ctx)
                for w in (
                    "first stays", "stays applied", "still applied", "keeps the first",
                    "first one stays", "first coupon stays", "leaves the existing",
                    "existing coupon untouched", "first remains", "already applied stays",
                    "first coupon already applied stays",
                )
            ),
        ),
        Check("catches the same-campaign guard (only in a PM comment)", lambda ctx: "campaign" in _plan(ctx)),
        Check(
            "notes the campaign guard applies ACROSS coupon types",
            lambda ctx: "campaign" in _plan(ctx)
            and any(
                w in _plan(ctx)
                for w in ("even if", "even when", "regardless", "different type", "both types", "across type", "different types")
            ),
        ),
        Check(
            "handles the negative-total edge (fixed > discounted subtotal)",
            lambda ctx: any(
                w in _plan(ctx)
                for w in (
                    "negative", "below zero", "below 0", "not go below", "never go below", "floor",
                    "clamp", "max(0", ">= 0", "≥ 0", "can't go negative", "cannot go negative",
                    "never go negative", "not go negative", "no lower than 0",
                )
            ),
        ),
        Check(
            "scopes the frontend badge OUT",
            lambda ctx: "frontend" in _plan(ctx)
            and any(w in _plan(ctx) for w in ("out of scope", "separate", "own branch", "not a gap")),
        ),
        Check(
            "plans the real verification signals",
            lambda ctx: "php -l" in _plan(ctx) and "php -r" in _plan(ctx) and ("check-task" in _plan(ctx) or "browser" in _plan(ctx)),
        ),
        Check("targets the full <id>/<slug> feature branch", lambda ctx: "61234/cart-coupon-stacking" in _plan(ctx)),
        Check(
            "does NOT propose a pull request (cherry-pick only)",
            lambda ctx: not any(
                phrase in _plan(ctx)
                for phrase in (
                    "open a pull request", "open a pr", "opening a pull request",
                    "prepare a pull request", "create a pull request", "submit a pull request",
                    "raise a pull request", "pull request to master", "pr to master",
                )
            ),
        ),
    ],
)


## Recorded calls — what the skill actually did

2 request(s) hit the fake service.

In [ ]:
# calls.jsonl (recorded by the fake service)

GET    /issue_logs/61234
GET    /issue_logs/61234/comments

## Result — `green-loop` · 13/13

- ✅ read the task from the tracker
- ✅ read the comments before planning
- ✅ wrote plan.md
- ✅ definition of done covers combinable percentage + fixed
- ✅ gets the ordering rule right (percentage on the subtotal, then fixed)
- ✅ AC2: a rejected duplicate keeps the first coupon applied
- ✅ catches the same-campaign guard (only in a PM comment)
- ✅ notes the campaign guard applies ACROSS coupon types
- ✅ handles the negative-total edge (fixed > discounted subtotal)
- ✅ scopes the frontend badge OUT
- ✅ plans the real verification signals
- ✅ targets the full <id>/<slug> feature branch
- ✅ does NOT propose a pull request (cherry-pick only)

In [ ]:
# artifact-green-loop.md — the plan this version produced
# Task #61234 — Coupon stacking: one percentage + one fixed coupon per cart

Branch: `61234/cart-coupon-stacking` (off master8). Column: In Progress — validator skeleton already committed by dev, so this is *finish the remaining work*, not a rewrite. Backend only.

## Definition of done

Derived strictly from the Expected result + the comments. Each line is a testable acceptance condition.

**From the acceptance criteria (Expected result):**
1. A cart may hold **at most one percentage coupon AND at most one fixed-amount coupon at the same time** (i.e. one of each, combinable).
2. A **second coupon of the same type is rejected with a clear error, and the already-applied coupon of that type stays applied** — the current silent-replace behaviour is a bug and must be removed, not preserved.
3. Application order is deterministic: **the percentage is applied to the subtotal FIRST, then the fixed amount is subtracted from that discounted subtotal.**

**From the comments (required additions / edges):**
4. (PM) **Same-campaign guard:** two coupons that share the same campaign id must be rejected even when their types differ (percentage + fixed of the *same* campaign is not allowed) — prevents double-dipping one campaign. This combines with #1: the two stacked coupons must be different type AND different campaign.
5. (dev edge) **Total floor:** a fixed-amount coupon can exceed the discounted subtotal, so the **final cart total must never go negative — it is floored at 0** (and the fixed discount effectively caps at the remaining amount).

**Explicitly OUT of scope (do not implement):**
- Max-5-coupons-per-cart cap — PM withdrew it; ignore.
- Cart-UI badge listing both coupons — separate FRONTEND task on its own branch; not a backend gap, not counted.

**Remaining work (per dev comment, skeleton exists):** the ordering rule (#3, #5) and the same-campaign guard (#4). Still verify #1/#2 in the skeleton actually *reject-and-keep-first* rather than replace.

## Plan

### 0. Setup / orientation (read-only first)
- Confirm HEAD is on `61234/cart-coupon-stacking` and it is based on master8. Do NOT commit or push master8/dev at any point. Delivery to dev/prod later is **cherry-pick only — never open, prepare, or suggest a PR.**
- Read the branch's committed diff to see what the "validator skeleton" already does: where coupons are validated on add-to-cart, the coupon model, and where cart totals are computed. Locate the coupon type field (percentage vs fixed) and the campaign id field.
- Use the **beliani-db-schema** MCP (prod, dev secondary) to confirm the real coupon/campaign tables and columns — coupon type discriminator, amount/percentage columns, campaign id FK, and the cart↔coupon link (how many coupons a cart can reference). No guessing table/column names from old code. Structure only.
- Before writing/altering any PHP or SQL, apply the **beliani-code-style** rules (avoid LIMIT 1 / verify the real key, PEAR::isError on the very next line, 3+ field updates through the model, complete phpDoc).

### 1. Implement the remaining rules
- **Type limit + reject-not-replace (#1, #2):** ensure adding a coupon checks the cart's already-applied coupons; if one of the same type exists, return a clear error and leave the existing one untouched. Kill any code path that silently overwrites the first coupon.
- **Same-campaign guard (#4):** on add, reject if the incoming coupon's campaign id matches any already-applied coupon's campaign id, regardless of type. Distinct, clear error message vs the same-type error.
- **Ordering + floor (#3, #5):** compute discounted subtotal = subtotal − (subtotal × pct); then total = discounted subtotal − fixed; floor at 0 so total (and the effective fixed discount) can never be negative. Ordering must be deterministic and independent of the order the two coupons were added.

### 2. Verification I will run (before calling deploy-ready)
- `php -l` on every changed PHP file.
- A `php -r` harness exercising the task's real examples + the comment-flagged edges, asserting exact numbers:
  - One percentage + one fixed (different campaigns) both apply. e.g. subtotal 100, 20% + 30 fixed → 100×0.8 = 80, 80 − 30 = **50** (proves percentage-before-fixed; reversed order would give 56).
  - **Negative floor:** subtotal 100, 20% + 90 fixed → 80 − 90 → floored to **0**, not −10.
  - **Same type rejected, first kept:** apply 10%, then attempt 15% → second rejected with error, cart still carries the 10%.
  - **Same campaign, different types rejected:** percentage(campaign 7) + fixed(campaign 7) → second rejected; percentage(campaign 7) + fixed(campaign 9) → both apply.
  - Confirm no silent replacement anywhere.
- Use **beliani-db-schema** / **beliani-sql-query** (dev) as needed to confirm the coupon type + campaign id columns and to sanity-check with a couple of real coupon rows that the type/campaign discriminators behave as assumed. Read-only.
- Browser QA via **check-task** on the branch environment: add a percentage coupon, then a fixed one → both stick; add a second percentage → clear error, first stays; add a fixed sharing the first's campaign → clear error; drive a cart where the fixed exceeds the discounted subtotal and confirm the total shows 0, not negative.

### 3. Deploy-ready
- When every DoF line above is verified green, record the task note in `docs/RELEASE_NOTES_backend.md`, including a short hands-on QA instruction (the coupon-stacking scenarios above) for the manager/QA final acceptance, and noting the cart-badge frontend piece lands from its own branch.
- Delivery to dev/prod is cherry-pick only; no PR.

### Loop / bounds
Execute → verify → fix, repeating only on a failing check, bounded by the attempt cap and no-progress guard. Frontend badge stays out of scope and is never treated as a gap.


## A/B scorecard

In [ ]:
# compare()

check                                                                 | green-loop
AC2: a rejected duplicate keeps the first coupon applied              | OK
catches the same-campaign guard (only in a PM comment)                | OK
definition of done covers combinable percentage + fixed               | OK
does NOT propose a pull request (cherry-pick only)                    | OK
gets the ordering rule right (percentage on the subtotal, then fixed) | OK
handles the negative-total edge (fixed > discounted subtotal)         | OK
notes the campaign guard applies ACROSS coupon types                  | OK
plans the real verification signals                                   | OK
read the comments before planning                                     | OK
read the task from the tracker                                        | OK
scopes the frontend badge OUT                                         | OK
targets the full <id>/<slug> feature branch                           | OK
wrote plan.md    